# Gestion de stocks – MDP\n
État : nombre d’unités en stock (0,1,2). Action : quantité commandée.

In [1]:
import numpy as np

In [3]:
def PolicyEvaluation(MDP, pi, gamma=0.9, theta=1e-10):
    states, actions, T, R = MDP
    n = len(states)
    V = np.zeros(n)
    while True:
        delta = 0
        for s in range(n):
            a = pi[s]
            v = sum(T[s][a][s2] * (R[s][a][s2] + gamma * V[s2]) for s2 in range(n))
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < theta:
            break
    return V

def ValueIteration(MDP, gamma=0.9, theta=1e-10):
    states, actions, T, R = MDP
    n = len(states)
    V = np.zeros(n)
    while True:
        delta = 0
        for s in range(n):
            vals = [sum(T[s][a][s2] * (R[s][a][s2] + gamma * V[s2]) for s2 in range(n))
                    for a in actions[s]]
            best = max(vals)
            delta = max(delta, abs(best - V[s]))
            V[s] = best
        if delta < theta:
            break
    pi = {}
    for s in range(n):
        vals = [sum(T[s][a][s2] * (R[s][a][s2] + gamma * V[s2]) for s2 in range(n))
                for a in actions[s]]
        pi[s] = actions[s][int(np.argmax(vals))]
    return V, pi

def PolicyIteration(MDP, gamma=0.9):
    states, actions, T, R = MDP
    n = len(states)
    pi = {s: actions[s][0] for s in range(n)}
    while True:
        V = PolicyEvaluation(MDP, pi, gamma)
        stable = True
        for s in range(n):
            old = pi[s]
            vals = [sum(T[s][a][s2] * (R[s][a][s2] + gamma * V[s2]) for s2 in range(n))
                    for a in actions[s]]
            pi[s] = actions[s][int(np.argmax(vals))]
            if pi[s] != old:
                stable = False
        if stable:
            break
    return V, pi

In [4]:
def build_stock():
    C, N, c_cost, h, v = 2, 2, 2, 3, 6
    states  = list(range(C+1))
    n       = C + 1
    actions = {i: list(range(C-i+1)) for i in states}
    Q = np.array([[0.4,0.1,0.0],[0.3,0.4,0.4],[0.3,0.5,0.6]])

    T_arr = [[[0.0]*n for _ in range(C+1)] for _ in range(n)]
    R_arr = [[[0.0]*n for _ in range(C+1)] for _ in range(n)]

    for i in states:
        for a in actions[i]:
            Y = i + a
            exp_r = 0
            for d in range(N+1):
                prob = Q[d][Y] if Y < n and d < n else 0
                exp_r += prob * (v*min(d,Y) - h*max(0,Y-d) - c_cost*a)
            for j in states:
                prob_j = sum(Q[d][Y] for d in range(N+1)
                             if Y < n and d < n and max(0,Y-d) == j)
                T_arr[i][a][j] = prob_j
                R_arr[i][a][j] = exp_r

    return (states, actions, T_arr, R_arr)

In [5]:
mdp_stock = build_stock()
V_s, pi_s   = ValueIteration(mdp_stock, gamma=0.9)
V_s2, pi_s2 = PolicyIteration(mdp_stock, gamma=0.9)

print("=== Gestion de stocks ===")
print("Value Iteration:")
for s in range(3):
    print(f"  V*(stock={s}) = {V_s[s]:.2f}  -> commander {pi_s[s]} unite(s)")
print("Policy Iteration:")
for s in range(3):
    print(f"  V*(stock={s}) = {V_s2[s]:.2f}  -> commander {pi_s2[s]} unite(s)")

=== Gestion de stocks ===
Value Iteration:
  V*(stock=0) = 51.20  -> commander 2 unite(s)
  V*(stock=1) = 53.20  -> commander 1 unite(s)
  V*(stock=2) = 55.20  -> commander 0 unite(s)
Policy Iteration:
  V*(stock=0) = 51.20  -> commander 2 unite(s)
  V*(stock=1) = 53.20  -> commander 1 unite(s)
  V*(stock=2) = 55.20  -> commander 0 unite(s)
